# Multi-Agent Healthcare Chatbot with AG2 (AutoGen)

A consultation assistant built as a **group of specialized agents** rather than one model. A patient agent describes symptoms, a diagnosis agent interprets them, and further specialists weigh in — all coordinated by a `GroupChatManager` that decides who speaks next.

The point is the orchestration pattern: several narrow agents in one conversation, each with its own remit.

> **Not medical advice.** This demonstrates a multi-agent architecture. The output is LLM-generated and can be wrong — consult a qualified clinician for anything real.

## Contents

1. Setup
2. Agents and their roles
3. Group chat and the manager
4. Running a consultation
5. A second crew: emotional wellbeing

## Setup


Libraries: `ag2` (AutoGen) for the agents, `python-dotenv` for the API key.

In [ ]:
%pip install -q 'ag2[openai]<1.0' python-dotenv

In [ ]:
import logging
import warnings

warnings.filterwarnings("ignore", category=DeprecationWarning)
warnings.filterwarnings("ignore", category=UserWarning)

from autogen import ConversableAgent, GroupChat, GroupChatManager

# Shared model configuration (see config.py); reads GROQ_API_KEY from .env.
from config import llm_config

# AG2 warns about key format when using a non-OpenAI endpoint.
logging.getLogger("autogen.oai.client").setLevel(logging.ERROR)

# Agents here only talk; no code execution, so Docker is not needed.
code_execution_config = {"use_docker": False}

print("Setup complete. Model:", llm_config["config_list"][0]["model"])

## What AG2 gives you

Agents each hold a role and a system message; a `GroupChat` puts them in one conversation and a `GroupChatManager` picks the next speaker. That turns a single prompt into a division of labour.

## The agents

Each one owns a narrow part of the consultation.

### ConversableAgent

The base agent: a name, a system message, and a model config.

In [ ]:

# Step 1: Create AI Agents with Defined Roles
patient_agent = ConversableAgent(
    name="patient", 
    system_message="You describe symptoms and ask for medical help.", 
    llm_config=llm_config
)

diagnosis_agent = ConversableAgent(
    name="diagnosis", 
    system_message="You analyze symptoms and provide a possible diagnosis. Summarize key points in one response.", 
    llm_config=llm_config
)

pharmacy_agent = ConversableAgent(
    name="pharmacy", 
    system_message="You recommend medications based on diagnosis. Only respond once.", 
    llm_config=llm_config
)

consultation_agent = ConversableAgent(
    name="consultation", 
    system_message="You determine if a doctor's visit is required. Provide a final summary with clear next steps. IMPORTANT: End your response with 'CONSULTATION_COMPLETE' to signal the end of the conversation.", 
    llm_config=llm_config
)


### GroupChat

Holds the agents and the shared transcript, capped by `max_round`.

In [ ]:
# Step 2: Create GroupChat for Structured Interaction
groupchat = GroupChat(
    agents=[diagnosis_agent, pharmacy_agent, consultation_agent],  # Patient only initiates
    messages=[], 
    max_round=5,  # Limits conversation to 5 rounds
    speaker_selection_method="round_robin"  # Ensures structured conversation flow
)

### GroupChatManager

Decides who speaks next based on the conversation so far.

In [ ]:
# Step 3: Create GroupChatManager to Handle Conversation
manager = GroupChatManager(name="manager", groupchat=groupchat)

## Run a consultation

The patient agent opens the conversation and the manager routes it from there.

In [ ]:
def consult(symptoms: str):
    """Run one consultation through the agent group chat."""
    print("\nDiagnosing symptoms...\n")
    return patient_agent.initiate_chat(
        manager,
        message=f"I am feeling {symptoms}. Can you help?",
    )


# Try it. Replace the text with whatever you want to route through the agents.
result = consult("a sore throat and a mild fever for two days")

## A second crew: emotional wellbeing

The same pattern with different specialists — showing that the structure, not the domain, is what carries over.

In [ ]:
from autogen import ConversableAgent, GroupChat, GroupChatManager

# Same pattern, different specialists: emotional support rather than diagnosis.
patient = ConversableAgent(
    name="patient",
    system_message="You describe your emotions and mental health concerns.",
    llm_config=llm_config,
)

emotion_analysis_agent = ConversableAgent(
    name="emotion_analysis",
    system_message=(
        "You identify the emotions expressed and reflect them back clearly and "
        "without judgement. Keep it to a few sentences."
    ),
    llm_config=llm_config,
)

coping_strategy_agent = ConversableAgent(
    name="coping_strategy",
    system_message=(
        "You suggest general, evidence-informed coping strategies such as grounding "
        "exercises, journaling or physical activity. You are not a clinician: "
        "encourage speaking to a qualified professional for anything serious."
    ),
    llm_config=llm_config,
)

wellbeing_chat = GroupChat(
    agents=[patient, emotion_analysis_agent, coping_strategy_agent],
    messages=[],
    max_round=4,
)
wellbeing_manager = GroupChatManager(groupchat=wellbeing_chat, llm_config=llm_config)

result = patient.initiate_chat(
    wellbeing_manager,
    message="I have been feeling overwhelmed and struggling to focus this week.",
)

## Author

**Anas AlGhannam**  
[github.com/AnasAlghannam](https://github.com/AnasAlghannam)